In [3]:
import os
import sys
import logging
from abc import ABC, abstractmethod
from typing import Dict, Any, Tuple, Optional

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("DiseasePredictionSystem")


# Custom Exception Hierarchy
class MedicalDataError(Exception):
    """Base exception for medical prediction workflow errors."""
    pass


class DatasetDownloadError(MedicalDataError):
    """Raised when remote or fallback dataset fetching fails."""
    pass


class DataPreprocessingError(MedicalDataError):
    """Raised during transformation, scaling, or splitting failures."""
    pass


class ModelTrainingError(MedicalDataError):
    """Raised when model fitting or cross-validation fails."""
    pass


# Data Loaders
class BaseDataLoader(ABC):
    """Abstract interface defining the extraction and loading contract."""

    def __init__(self, dataset_name: str, primary_url: Optional[str] = None):
        self.dataset_name = dataset_name
        self.primary_url = primary_url

    @abstractmethod
    def load_data(self) -> Tuple[pd.DataFrame, pd.Series]:
        """Loads features (X) and target labels (y)."""
        pass

    def _fetch_from_url(self, url: str, **read_csv_kwargs) -> pd.DataFrame:
        """Helper to download and parse CSV datasets safely."""
        try:
            logger.info(f"Fetching {self.dataset_name} data from {url}")
            df = pd.read_csv(url, **read_csv_kwargs)
            if df.empty:
                raise DatasetDownloadError(f"Downloaded dataset from {url} is empty.")
            return df
        except Exception as e:
            raise DatasetDownloadError(f"Failed to load dataset from {url}. Reason: {e}")


class HeartDiseaseLoader(BaseDataLoader):
    """Loads UCI Heart Disease Dataset (Cleveland)."""

    def __init__(self):
        super().__init__(
            dataset_name="Heart Disease (UCI Cleveland)",
            primary_url="https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
        )
        self.column_names = [
            "age", "sex", "cp", "trestbps", "chol", "fbs",
            "restecg", "thalach", "exang", "oldpeak", "slope",
            "ca", "thal", "target"
        ]

    def load_data(self) -> Tuple[pd.DataFrame, pd.Series]:
        try:
            df = self._fetch_from_url(
                self.primary_url,
                names=self.column_names,
                na_values="?"
            )
        except DatasetDownloadError:
            logger.warning("Primary UCI mirror failed. Switching to reliable GitHub CDN mirror.")
            mirror_url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/heart.csv"
            df = self._fetch_from_url(mirror_url)

        # Standardize target to binary (0 = No Disease, 1 = Disease present)
        target_col = "target" if "target" in df.columns else df.columns[-1]
        df[target_col] = (df[target_col] > 0).astype(int)

        X = df.drop(columns=[target_col])
        y = df[target_col]
        return X, y


class DiabetesLoader(BaseDataLoader):
    """Loads Pima Indians Diabetes Dataset."""

    def __init__(self):
        super().__init__(
            dataset_name="Pima Indians Diabetes",
            primary_url="https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
        )
        self.column_names = [
            "pregnancies", "glucose", "blood_pressure", "skin_thickness",
            "insulin", "bmi", "diabetes_pedigree", "age", "outcome"
        ]

    def load_data(self) -> Tuple[pd.DataFrame, pd.Series]:
        try:
            df = self._fetch_from_url(self.primary_url, names=self.column_names)
        except DatasetDownloadError:
            logger.warning("Primary diabetes URL unreachable. Attempting secondary mirror.")
            mirror_url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
            df = self._fetch_from_url(mirror_url)
            df.columns = [col.lower() for col in df.columns]

        target_col = "outcome"
        # Biologically impossible zeroes treated as NaN
        zero_invalid_cols = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]
        for col in zero_invalid_cols:
            if col in df.columns:
                df[col] = df[col].replace(0, np.nan)

        X = df.drop(columns=[target_col])
        y = df[target_col].astype(int)
        return X, y


class BreastCancerLoader(BaseDataLoader):
    """Loads Breast Cancer Wisconsin (Diagnostic) Dataset."""

    def __init__(self):
        super().__init__(
            dataset_name="Breast Cancer Wisconsin",
            primary_url="https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
        )

    def load_data(self) -> Tuple[pd.DataFrame, pd.Series]:
        # Using Scikit-Learn's built-in UCI WDBC loader as first-line source to guarantee integrity
        try:
            raw_data = load_breast_cancer(as_frame=True)
            X = raw_data.data
            y = raw_data.target  # 0: Malignant, 1: Benign
            return X, y
        except Exception as e:
            logger.warning(f"Native loader encountered error: {e}. Attempting direct UCI remote link.")
            df = self._fetch_from_url(self.primary_url, header=None)
            X = df.iloc[:, 2:]
            y = df.iloc[:, 1].map({"M": 1, "B": 0}).astype(int)
            return X, y


# Preprocessing Pipeline
class MedicalDataPipeline:
    """Handles missing value imputation, robust feature scaling, and train/test splits."""

    def __init__(self, test_size: float = 0.2, random_state: int = 42):
        self.test_size = test_size
        self.random_state = random_state
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()
        self.feature_names = []

    def fit_transform(
        self, X: pd.DataFrame, y: pd.Series
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        try:
            self.feature_names = list(X.columns)

            # Prevent data leakage: Split before fit_transform
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=self.test_size,
                random_state=self.random_state,
                stratify=y
            )

            # Impute & Scale on Train, then Transform Test
            X_train_imp = self.imputer.fit_transform(X_train)
            X_test_imp = self.imputer.transform(X_test)

            X_train_scaled = self.scaler.fit_transform(X_train_imp)
            X_test_scaled = self.scaler.transform(X_test_imp)

            return X_train_scaled, X_test_scaled, y_train.to_numpy(), y_test.to_numpy()

        except Exception as e:
            raise DataPreprocessingError(f"Error during preprocessing pipeline: {e}")

    def transform_single_instance(self, instance: pd.DataFrame) -> np.ndarray:
        """Transforms a single incoming patient record for runtime inference."""
        try:
            instance_imp = self.imputer.transform(instance)
            return self.scaler.transform(instance_imp)
        except Exception as e:
            raise DataPreprocessingError(f"Failed to preprocess inference instance: {e}")


# Model Training & Inference (OOP: Polymorphism & Strategy)
class DiseaseModelTrainer:
    """Manages training, cross-validation, and performance metrics across algorithms."""

    def __init__(self, random_state: int = 42):
        self.random_state = random_state
        self.models: Dict[str, Any] = {
            "Logistic Regression": LogisticRegression(
                max_iter=1000,
                random_state=self.random_state
            ),
            "Support Vector Machine": SVC(
                probability=True,
                kernel="rbf",
                random_state=self.random_state
            ),
            "Random Forest": RandomForestClassifier(
                n_estimators=150,
                max_depth=6,
                random_state=self.random_state
            ),
            "XGBoost": XGBClassifier(
                n_estimators=120,
                learning_rate=0.05,
                max_depth=4,
                eval_metric="logloss",
                random_state=self.random_state
            )
        }
        self.trained_models: Dict[str, Any] = {}
        self.performance_summary: Dict[str, Dict[str, float]] = {}

    def train_and_evaluate(
        self,
        X_train: np.ndarray,
        X_test: np.ndarray,
        y_train: np.ndarray,
        y_test: np.ndarray
    ) -> Dict[str, Dict[str, float]]:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.random_state)

        for name, model in self.models.items():
            try:
                # 5-fold cross validation on training set
                cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
                model.fit(X_train, y_train)
                self.trained_models[name] = model

                y_pred = model.predict(X_test)
                y_prob = model.predict_proba(X_test)[:, 1]

                self.performance_summary[name] = {
                    "CV_ROC_AUC": float(np.mean(cv_scores)),
                    "Accuracy": float(accuracy_score(y_test, y_pred)),
                    "Precision": float(precision_score(y_test, y_pred, zero_division=0)),
                    "Recall": float(recall_score(y_test, y_pred, zero_division=0)),
                    "F1_Score": float(f1_score(y_test, y_pred, zero_division=0)),
                    "Test_ROC_AUC": float(roc_auc_score(y_test, y_prob))
                }
                logger.info(f"Model successfully trained: {name}")

            except Exception as e:
                logger.error(f"Failed to train {name}: {e}")
                raise ModelTrainingError(f"Training failed on algorithm: {name}")

        return self.performance_summary

    def get_best_model(self, metric: str = "Test_ROC_AUC") -> Tuple[str, Any]:
        """Returns the best model based on the selected metric."""
        if not self.performance_summary:
            raise ModelTrainingError("No trained models found. Train models first.")

        best_model_name = max(
            self.performance_summary,
            key=lambda k: self.performance_summary[k][metric]
        )
        return best_model_name, self.trained_models[best_model_name]


# Prediction Controller / Application Orchestrator
class MedicalPredictorApp:
    """Master controller that wires together loaders, pipelines, and models."""

    def __init__(self, data_loader: BaseDataLoader):
        self.loader = data_loader
        self.pipeline = MedicalDataPipeline()
        self.trainer = DiseaseModelTrainer()

    def run(self):
        print(f"\n{'='*70}\nRunning Pipeline: {self.loader.dataset_name}\n{'='*70}")
        try:
            # Load Data
            X, y = self.loader.load_data()
            print(f"Dataset Loaded Successfully: {X.shape[0]} samples, {X.shape[1]} features.")

            # Preprocess & Split
            X_train, X_test, y_train, y_test = self.pipeline.fit_transform(X, y)

            # Train & Evaluate Models
            results = self.trainer.train_and_evaluate(X_train, X_test, y_train, y_test)
            results_df = pd.DataFrame(results).T
            print("\nPerformance Comparison Matrix:")
            print(results_df.round(4).to_string())

            # Identify Best Performing Model
            best_name, best_model = self.trainer.get_best_model(metric="Test_ROC_AUC")
            print(f"\nChampion Model: {best_name} (Highest ROC-AUC: {results[best_name]['Test_ROC_AUC']:.4f})")

            # Example Prediction
            sample_patient = X.iloc[[0]]
            processed_sample = self.pipeline.transform_single_instance(sample_patient)
            prediction = best_model.predict(processed_sample)[0]
            confidence = best_model.predict_proba(processed_sample)[0][prediction]

            status = "Disease Detected" if prediction == 1 else "Normal / No Disease"
            print(f"Sample Inference -> Result: {status} (Confidence: {confidence:.2%})")

        except MedicalDataError as mde:
            logger.critical(f"Medical Pipeline Error: {mde}")
        except Exception as ex:
            logger.critical(f"Unexpected Fatal Error: {ex}", exc_info=True)

# User Prediction System

class UserInputValidationError(MedicalDataError):
    """Raised when user input fails numerical or physical bounds validation."""
    pass


class UserPredictionSystem:
    """
    Interactive OOP orchestrator for capturing user/patient medical inputs,
    validating feature ranges, and running real-time diagnostic predictions.
    """

    def __init__(self, predictor_app: MedicalPredictorApp):
        self.app = predictor_app
        self.pipeline = predictor_app.pipeline
        self.trainer = predictor_app.trainer
        self.feature_names = predictor_app.pipeline.feature_names

    def _prompt_feature_value(self, feature: str, default_val: float) -> float:
        """Prompts for an individual feature value with recursive validation and graceful fallback."""
        while True:
            try:
                user_val = input(f" Enter value for '{feature}' [Default: {default_val:.2f}]: ").strip()
                if not user_val:
                    return float(default_val)
                val = float(user_val)
                if np.isnan(val) or np.isinf(val):
                    raise ValueError("Value cannot be NaN or Infinite.")
                return val
            except ValueError as ve:
                print(f"   [!] Invalid numerical input: {ve}. Please enter a valid number.")

    def collect_patient_data(self) -> pd.DataFrame:
        """Collects and validates inputs for all expected pipeline features."""
        if not self.feature_names:
            raise UserInputValidationError("Underlying model pipeline has no fitted feature names.")

        print(f"\n--- Patient Input Collection: {self.app.loader.dataset_name} ---")
        print(f"Total features required: {len(self.feature_names)}")
        print("Tip: Press ENTER without typing to use dataset median default values.\n")

        patient_records: Dict[str, float] = {}
        # Use imputer medians as baseline reference defaults
        default_values = self.pipeline.imputer.statistics_

        for idx, feature in enumerate(self.feature_names):
            default_val = float(default_values[idx]) if idx < len(default_values) else 0.0
            patient_records[feature] = self._prompt_feature_value(feature, default_val)

        return pd.DataFrame([patient_records])

    def predict_for_user(self):
        """Runs end-to-end interactive prediction with model selection and exception recovery."""
        try:
            best_name, best_model = self.trainer.get_best_model(metric="Test_ROC_AUC")
            patient_df = self.collect_patient_data()

            # Preprocess via the trained pipeline
            processed_patient = self.pipeline.transform_single_instance(patient_df)

            # Predict outcome and class probabilities
            prediction = best_model.predict(processed_patient)[0]
            probabilities = best_model.predict_proba(processed_patient)[0]

            # Interpret label based on specific dataset semantics
            if "Breast Cancer" in self.app.loader.dataset_name:
                # Scikit-learn WDBC: 0 = Malignant, 1 = Benign
                status = "Malignant (Cancer Detected)" if prediction == 0 else "Benign (No Cancer)"
                confidence = probabilities[prediction]
            else:
                status = "Disease Detected / Positive Risk" if prediction == 1 else "Normal / Low Risk"
                confidence = probabilities[prediction]

            print("\n" + "=" * 55)
            print("         PATIENT DIAGNOSTIC REPORT")
            print("=" * 55)
            print(f" Dataset Scope      : {self.app.loader.dataset_name}")
            print(f" Classifier Engine  : {best_name}")
            print(f" Diagnosis Result   : {status}")
            print(f" Confidence Score   : {confidence:.2%}")
            print(f" Probability Vector : Class 0: {probabilities[0]:.2%}, Class 1: {probabilities[1]:.2%}")
            print("=" * 55 + "\n")

        except UserInputValidationError as uve:
            logger.error(f"Input validation error: {uve}")
        except DataPreprocessingError as dpe:
            logger.error(f"Preprocessing failed during inference: {dpe}")
        except Exception as e:
            logger.error(f"Inference execution failed: {e}", exc_info=True)


# Execution Entry Point
if __name__ == "__main__":
    # 1. Run Heart Disease Classification
    heart_app = MedicalPredictorApp(HeartDiseaseLoader())
    heart_app.run()

    # 2. Run Diabetes Classification
    diabetes_app = MedicalPredictorApp(DiabetesLoader())
    diabetes_app.run()

    # 3. Run Breast Cancer Classification
    cancer_app = MedicalPredictorApp(BreastCancerLoader())
    cancer_app.run()

    # 4. Interactive User Prediction System
    print("\n" + "#" * 70)
    print("      INTERACTIVE USER MEDICAL PREDICTION SYSTEM")
    print("#" * 70)

    menu = {
        "1": (heart_app, "Heart Disease Predictor"),
        "2": (diabetes_app, "Diabetes Predictor"),
        "3": (cancer_app, "Breast Cancer Predictor")
    }

    while True:
        print("\nSelect an application to test patient data:")
        for key, (_, name) in menu.items():
            print(f" [{key}] {name}")
        print(" [4] Exit")

        choice = input("Enter choice (1-4): ").strip()
        if choice == "4":
            print("Exiting prediction system. Stay healthy!")
            break
        elif choice in menu:
            selected_app, _ = menu[choice]
            user_system = UserPredictionSystem(selected_app)
            user_system.predict_for_user()
        else:
            print("[!] Invalid selection. Please enter 1, 2, 3, or 4.")


Running Pipeline: Heart Disease (UCI Cleveland)
2026-10-19 14:08:44,506 [INFO] DiseasePredictionSystem: Fetching Heart Disease (UCI Cleveland) data from https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data
Dataset Loaded Successfully: 303 samples, 13 features.
2026-10-19 14:08:45,804 [INFO] DiseasePredictionSystem: Model successfully trained: Logistic Regression
2026-10-19 14:08:45,880 [INFO] DiseasePredictionSystem: Model successfully trained: Support Vector Machine
2026-10-19 14:08:47,199 [INFO] DiseasePredictionSystem: Model successfully trained: Random Forest
2026-10-19 14:08:47,371 [INFO] DiseasePredictionSystem: Model successfully trained: XGBoost

Performance Comparison Matrix:
                        CV_ROC_AUC  Accuracy  Precision  Recall  F1_Score  Test_ROC_AUC
Logistic Regression         0.8960    0.8689     0.8125  0.9286    0.8667        0.9513
Support Vector Machine      0.8828    0.8525     0.8065  0.8929    0.8475        0.94

Enter choice (1-4):  1



--- Patient Input Collection: Heart Disease (UCI Cleveland) ---
Total features required: 13
Tip: Press ENTER without typing to use dataset median default values.



 Enter value for 'age' [Default: 56.00]:  18
 Enter value for 'sex' [Default: 1.00]:  1
 Enter value for 'cp' [Default: 3.00]:  2
 Enter value for 'trestbps' [Default: 130.00]:  100
 Enter value for 'chol' [Default: 244.50]:  200
 Enter value for 'fbs' [Default: 0.00]:  1
 Enter value for 'restecg' [Default: 0.00]:  0
 Enter value for 'thalach' [Default: 153.50]:  100
 Enter value for 'exang' [Default: 0.00]:  1
 Enter value for 'oldpeak' [Default: 0.80]:  1.20
 Enter value for 'slope' [Default: 2.00]:  1
 Enter value for 'ca' [Default: 0.00]:  
 Enter value for 'thal' [Default: 3.00]:  



         PATIENT DIAGNOSTIC REPORT
 Dataset Scope      : Heart Disease (UCI Cleveland)
 Classifier Engine  : Random Forest
 Diagnosis Result   : Normal / Low Risk
 Confidence Score   : 72.56%
 Probability Vector : Class 0: 72.56%, Class 1: 27.44%


Select an application to test patient data:
 [1] Heart Disease Predictor
 [2] Diabetes Predictor
 [3] Breast Cancer Predictor
 [4] Exit


Enter choice (1-4):  4


Exiting prediction system. Stay healthy!
